<a href="https://colab.research.google.com/github/Man1ek27/SentimentAnalysisCUDA/blob/llm-optimization/Trump_Tweet_DistilGPT2_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trump Tweet Fine-Tuning on DistilGPT-2
Fine-tune DistilGPT-2 on Trump's Legacy dataset to generate Trump-style tweets.

In [1]:
# Install required packages
!pip install -q torch transformers datasets pandas kaggle

In [2]:
import torch
import pandas as pd
from pathlib import Path

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Set up Kaggle API from Colab Secrets
from google.colab import userdata
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Save credentials to ~/.kaggle/kaggle.json
import os
import json
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle API configured')

SecretNotFoundError: Secret KAGGLE_USERNAME does not exist.

In [4]:
# Download Trump's Legacy dataset from Kaggle
import subprocess
dataset_path = Path('/tmp/trump_data')
dataset_path.mkdir(exist_ok=True)

print('Downloading Trump\'s Legacy dataset...')
subprocess.run(['kaggle', 'datasets', 'download', '-d', 'zusmani/trumps-legacy', '-p', str(dataset_path)], check=True)

# Extract zip
import zipfile
zip_file = list(dataset_path.glob('*.zip'))[0]
with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(dataset_path)
print(f'Dataset extracted to {dataset_path}')
print('Files:', list(dataset_path.glob('*.csv'))[:3])

Dataset extracted to /tmp/trump_data
Files: [PosixPath('/tmp/trump_data/Trumps Legcy.csv')]


In [5]:
# Load tweets dataset
csv_files = list(dataset_path.glob('*.csv'))
df = pd.read_csv(csv_files[0])
print(f'Loaded {len(df)} tweets')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFirst 3 tweets:')
print(df['text'].head(3).values)

Loaded 56571 tweets
Columns: ['id', 'text', 'device', 'favorites', 'retweets', 'date']

First 3 tweets:
['Republicans and Democrats have both created our economic problems.'
 'I was thrilled to be back in the Great city of Charlotte, North Carolina with thousands of hardworking American Patriots who love our Country, cherish our values, respect our laws, and always put AMERICA FIRST! Thank you for a wonderful evening!! #KAG2020 https://t.co/dNJZfRsl9y'
 'RT @CBS_Herridge: READ: Letter to surveillance court obtained by CBS News questions where there will be further disciplinary action and cho…']


In [6]:
# Data exploration
print(f'Total tweets: {len(df)}')
print(f'Avg tweet length: {df["text"].str.len().mean():.0f} chars')
print(f'Min/Max length: {df["text"].str.len().min()}/{df["text"].str.len().max()}')
print(f'\nSample Trump tweets:')
for i, tweet in enumerate(df['text'].sample(3).values, 1):
    print(f'{i}. {tweet[:80]}...' if len(tweet) > 80 else f'{i}. {tweet}')

Total tweets: 56571
Avg tweet length: 128 chars
Min/Max length: 2/328

Sample Trump tweets:
1. RT @jasoninthehouse: Agreed
2. RT @BrandonStraka: I can’t even describe the sinking feeling I got last night be...
3. https://t.co/w9YruJ4AYc


In [7]:
# Clean tweets: remove URLs, @mentions, and extra whitespace
import re

def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_tweet)
# Remove empty tweets
df = df[df['text'].str.len() > 10]
print(f'After cleaning: {len(df)} tweets')
print(f'Sample cleaned tweet: {df["text"].iloc[0][:80]}...')

After cleaning: 53753 tweets
Sample cleaned tweet: Republicans and Democrats have both created our economic problems....


In [8]:
# Prepare tokenizer and encode tweets
from transformers import AutoTokenizer

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize all tweets
encodings = tokenizer(df['text'].tolist(), truncation=True, max_length=128, padding='max_length', return_tensors='pt')
print(f'Tokenized {len(encodings["input_ids"])} tweets')
print(f'Token shape: {encodings["input_ids"].shape}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenized 53753 tweets
Token shape: torch.Size([53753, 128])


In [ ]:
## Create train/validation split (80/20)
#from torch.utils.data import TensorDataset, DataLoader, random_split
#
#dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
#train_size = int(0.8 * len(dataset))
#val_size = len(dataset) - train_size
#train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#
## Create DataLoaders
#batch_size = 16
#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size)
#
#print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [9]:
# Load pre-trained DistilGPT-2
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f'Model loaded: {model_name}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
model.train()

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: distilgpt2
Parameters: 81.9M


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
## Training configuration
#from torch.optim import AdamW
#
#num_epochs = 3
#learning_rate = 5e-5
#optimizer = AdamW(model.parameters(), lr=learning_rate)
#
#print(f'Epochs: {num_epochs}, LR: {learning_rate}, Batch size: {batch_size}')

In [ ]:
## Fine-tuning loop
#import time
#
#best_val_loss = float('inf')
#checkpoint_dir = Path('/tmp/trump_model_checkpoints')
#checkpoint_dir.mkdir(exist_ok=True)
#
#for epoch in range(num_epochs):
#    # Training
#    train_loss = 0
#    model.train()
#    for batch_idx, (input_ids, attention_mask) in enumerate(train_loader):
#        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
#        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
#        loss = outputs.loss
#        loss.backward()
#        optimizer.step()
#        optimizer.zero_grad()
#        train_loss += loss.item()
#        if (batch_idx + 1) % 50 == 0:
#            print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}')
#
#    avg_train_loss = train_loss / len(train_loader)
#
#    # Validation
#    val_loss = 0
#    model.eval()
#    with torch.no_grad():
#        for input_ids, attention_mask in val_loader:
#            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
#            outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
#            val_loss += outputs.loss.item()
#
#    avg_val_loss = val_loss / len(val_loader)
#    print(f'Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}')
#
#    # Save best checkpoint
#    if avg_val_loss < best_val_loss:
#        best_val_loss = avg_val_loss
#        model.save_pretrained(checkpoint_dir / f'best_model')
#        print(f'  -> Saved best model (val loss: {avg_val_loss:.4f})')

In [ ]:
import torch
from torch.utils.data import Dataset, random_split
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Custom Dataset wrapper to format output items as dictionaries for the Trainer
class TrumpDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {key: val[idx].clone().detach() for key, val in self.encodings.items()}

# 2. Wrap the global encodings object into the compliant dataset structure
full_dataset = TrumpDataset(encodings)

# 3. Create train/validation split (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 4. Data collator for language modeling (handles batch formatting on GPU)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Fast training loop configuration
training_args = TrainingArguments(
    output_dir='/tmp/trump_model_checkpoints',
    num_train_epochs=3,                  # Number of epochs: 3
    per_device_train_batch_size=16,      # Batch size 16 for training
    per_device_eval_batch_size=16,       # Batch size 16 for evaluation
    learning_rate=5e-5,                  # Learning rate
    weight_decay=0.01,
    logging_steps=50,                    # Logging step interval
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,         # Automatically load the best model version at the end
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=True,                           # Enable mixed precision training for CUDA acceleration
    report_to="none"
)

# 6. Initialize training manager
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# 7. Start fine-tuning execution
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.590139,3.504111


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:

from transformers import AutoModelForCausalLM
from pathlib import Path
checkpoint_dir = Path('/tmp/trump_model_checkpoints')


best_model_path = checkpoint_dir / 'best_model'
model = AutoModelForCausalLM.from_pretrained(best_model_path).to(device)
# Reload the best saved model using from_pretrained (works with safetensors)
best_model_path = checkpoint_dir / 'best_model'
model = AutoModelForCausalLM.from_pretrained(best_model_path).to(device)

# Fix: explicitly set pad_token_id and configure generation properly
model.config.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token
model.eval()

# Generate text with different seed prompts
prompts = ['Make America', 'I will', 'The fake news', 'Beautiful', 'Total disaster']
print('Generated Trump-style tweets:\n')

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    output = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],  # pass attention mask
        max_new_tokens=50,                         # only generate new tokens, don't repeat prompt
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,       # explicit pad token
        repetition_penalty=1.1,                    # reduce repetition
        no_repeat_ngram_size=2                     # avoid repeating bigrams
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'Prompt: "{prompt}"')
    print(f'Generated: "{generated_text}"\n')

In [ ]:
## Optional: Compare perplexity before/after fine-tuning
#import math
#
#print('Perplexity Analysis (lower is better)\n')
#
## Fine-tuned model perplexity
#finetuned_loss = best_val_loss
#finetuned_ppl = math.exp(finetuned_loss)
#print(f'Fine-tuned model validation perplexity: {finetuned_ppl:.2f}')
#
## Load original DistilGPT-2 for comparison
#original_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
#original_model.eval()
#orig_val_loss = 0
#with torch.no_grad():
#    for input_ids, attention_mask in val_loader:
#        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
#        outputs = original_model(input_ids, attention_mask=attention_mask, labels=input_ids)
#        orig_val_loss += outputs.loss.item()
#orig_val_loss /= len(val_loader)
#orig_ppl = math.exp(orig_val_loss)
#
#print(f'Original DistilGPT-2 validation perplexity: {orig_ppl:.2f}')
#print(f'\nImprovement: {orig_ppl - finetuned_ppl:.2f} perplexity points ({(orig_ppl/finetuned_ppl - 1)*100:.1f}% reduction)')

In [ ]:
import math
from transformers import AutoModelForCausalLM, Trainer

print('Perplexity Analysis (lower is better)\n')

# perplexity form ours
finetuned_loss = trainer.evaluate()['eval_loss']
finetuned_ppl = math.exp(finetuned_loss)
print(f'Fine-tuned model validation perplexity: {finetuned_ppl:.2f}')

# original gpt distileld
original_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
original_model.eval()


baseline_trainer = Trainer(
    model=original_model,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

orig_val_loss = baseline_trainer.evaluate()['eval_loss']
orig_ppl = math.exp(orig_val_loss)

# results
print(f'Original GPT-2 validation perplexity: {orig_ppl:.2f}')
print(f'\nImprovement: {orig_ppl - finetuned_ppl:.2f} perplexity points ({(orig_ppl/finetuned_ppl - 1)*100:.1f}% reduction)')